In [13]:
import json
import os
import shutil
import yaml
from pathlib import Path
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from ultralytics import YOLO

import dagshub, mlflow, torch

In [14]:
os.environ['YOLO_MLFLOW'] = 'true'
dagshub.init(repo_owner = 'paezagustin', repo_name = 'ratitas_2026', mlflow=True)

Accessing as paezagustin

Initialized MLflow to track repo "paezagustin/ratitas_2026"

Repository paezagustin/ratitas_2026 initialized!

In [15]:
SOURCE_DIR = "/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/classif_256x256_newannot"
OUTPUT_DIR = "/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification"

CLASS_NAMES = ['Decidua', 'Labyrinth', 'Junctional Zone', 'no_labeled', 'background']
VERSION = "v1"

In [3]:
def convert_multi_to_single_label(classes):
    """
    Convierte multi-label a single-label.
    Prioridad: Decidua > Labyrinth > Junctional Zone > no_labeled > background
    """
    priority_order = [0, 1, 2, 3, 4]
    for idx in priority_order:
        if classes[idx] == 1:
            return idx
    return 4

def prepare_yolo_classification_data():
    splits = ['train', 'val', 'test']
    
    for split in splits:
        images_src = os.path.join(SOURCE_DIR, split, 'images')
        labels_src = os.path.join(SOURCE_DIR, split, 'labels')
        
        for class_name in CLASS_NAMES:
            os.makedirs(os.path.join(OUTPUT_DIR, split, class_name), exist_ok=True)
        
        if not os.path.exists(images_src):
            print(f"Skip {split} - no images directory")
            continue
        
        count = 0
        for img_file in os.listdir(images_src):
            if not img_file.endswith('.jpg'):
                continue

            img_path = os.path.join(images_src, img_file)
            label_file = img_file.replace('.jpg', '.json')
            label_path = os.path.join(labels_src, label_file)

            if not os.path.exists(label_path):
                continue

            with open(label_path, 'r') as f:
                label_data = json.load(f)
            
            classes = label_data['classes']
            single_label = convert_multi_to_single_label(classes)
            class_name = CLASS_NAMES[single_label]
            
            dest_path = os.path.join(OUTPUT_DIR, split, class_name, img_file)
            shutil.copy2(img_path, dest_path)
            count += 1
        
        print(f"{split}: {count} imágenes procesadas")

prepare_yolo_classification_data()

train: 46353 imágenes procesadas
val: 5424 imágenes procesadas
test: 5235 imágenes procesadas


In [16]:
data_yaml = {
    'path': OUTPUT_DIR,
    'train': 'train',
    'val': 'val',
    'test': 'test',
    'names': {i: name for i, name in enumerate(CLASS_NAMES)},
    'nc': len(CLASS_NAMES)
}

yaml_path = "/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification/dataset.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"YAML guardado en: {yaml_path}")
print("\nDistribución de clases:")
for split in ['train', 'val', 'test']:
    print(f"\n{split}:")
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(OUTPUT_DIR, split, class_name)
        count = len(os.listdir(class_dir)) if os.path.exists(class_dir) else 0
        print(f"  {class_name}: {count}")

YAML guardado en: /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification/dataset.yaml

Distribución de clases:

train:
  Decidua: 4440
  Labyrinth: 8852
  Junctional Zone: 2129
  no_labeled: 4601
  background: 26331

val:
  Decidua: 517
  Labyrinth: 1086
  Junctional Zone: 73
  no_labeled: 690
  background: 3058

test:
  Decidua: 725
  Labyrinth: 1397
  Junctional Zone: 216
  no_labeled: 306
  background: 2591


In [20]:
MODEL_NAME = "yolo26x-cls"
EPOCHS = 100
IMGSZ = 256
BATCH = 128
PROJECT = "lci_yolo_classifier"
NAME = 'yolo26x_256x256_'

model = YOLO(f'{MODEL_NAME}.pt')

results = model.train(
    data=OUTPUT_DIR,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name=NAME,
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
    device=0,
    workers=8
)

New https://pypi.org/project/ultralytics/8.4.32 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.13 🚀 Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3080 Ti, 12042MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tra

In [22]:
best_model_path = f"/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/scripts/training_classifier/runs/classify/lci_yolo_classifier/yolo26x_256x256_/weights/best.pt"
model = YOLO(best_model_path)

val_results = model.val()
print("\n=== Resultados de Validación ===")
print(f"Top-1 Accuracy: {val_results.top1}")
print(f"Top-5 Accuracy: {val_results.top5}")

Ultralytics 8.4.13 🚀 Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3080 Ti, 12042MiB)
YOLO26x-cls summary (fused): 94 layers, 28,338,821 parameters, 0 gradients, 110.3 GFLOPs
train: /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification/train... found 46353 images in 5 classes ✅ 
val: /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification/val... found 5424 images in 5 classes ✅ 
test: /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification/test... found 5235 images in 5 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 690.4±454.4 MB/s, size: 7.6 KB)
val: Scanning /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification/val... 5424 images, 0 corrupt: 100% ━━━━━━━━━━━━ 5424/5424 1.3Git/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 339/339 43.6it/s 7.8s0.1s
       

In [23]:
test_results = model.val(data=yaml_path, split='test')
print("\n=== Resultados en Test ===")
print(f"Top-1 Accuracy: {test_results.top1}")
print(f"Top-5 Accuracy: {test_results.top5}")

Ultralytics 8.4.13 🚀 Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3080 Ti, 12042MiB)
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 693.1±546.1 MB/s, size: 8.8 KB)
test: Scanning /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification/test... 5235 images, 0 corrupt: 100% ━━━━━━━━━━━━ 5235/5235 9.7Kit/s 0.5s0.0s
test: New cache created: /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/yolo_classification/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 328/328 44.0it/s 7.5s0.0s
                   all      0.908          1
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/scripts/training_classifier/runs/classify/val2

=== Resultados en Test ===
Top-1 Accuracy: 0.9077363610267639
Top-5 Accuracy: 1.0


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

test_images_dir = os.path.join(OUTPUT_DIR, 'test')
sample_images = []

for class_name in CLASS_NAMES:
    class_dir = os.path.join(test_images_dir, class_name)
    images = os.listdir(class_dir)[:1]
    for img_name in images:
        sample_images.append((os.path.join(class_dir, img_name), class_name))

fig, axes = plt.subplots(1, len(sample_images), figsize=(15, 3))
if len(sample_images) == 1:
    axes = [axes]

for idx, (img_path, true_class) in enumerate(sample_images):
    img = Image.open(img_path)
    result = model.predict(img_path, verbose=False)[0]
    pred_class = result.names[result.probs.top1]
    confidence = result.probs.top1conf.item()
    
    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(f"True: {true_class}\nPred: {pred_class} ({confidence:.2f})")

plt.tight_layout()
plt.savefig(f"{PROJECT}/{NAME}/sample_predictions.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Predicciones guardadas en: {PROJECT}/{NAME}/sample_predictions.png")